# 🎙️ Kashmiri ASR Fine-Tuning: Meta Wav2Vec2-BERT 2.0 (`facebook/w2v-bert-2.0`)
**Google Colab Pro Runner** — Clone Repo, Load from **Hugging Face Hub** OR **Google Drive Shards**, Train/Resume & Evaluate

---

### 🛡️ Dual Dataset Source + Disconnect Resilience
- **Hugging Face Hub OR Google Drive Shards**: Set `DATASET_SOURCE` in **Cell 2** to:
  - `"huggingface"` to train on `munzirahangar/kas-news-dataset` (or any HF dataset ID), OR
  - `"gdrive"` to concatenate saved Arrow shards (`<DRIVE_DATASET_DIR>/shards/batch_*` with `state.json`) directly from Google Drive using `load_from_disk` + `concatenate_datasets` + `cast_column("audio", Audio(sampling_rate=16000))`.
- **Isolated Checkpoints per Dataset**: Each dataset automatically gets its own checkpoint and vocabulary subfolder on Google Drive so switching between `kas-news-dataset` and `voicehub_dataset` never collides.
- **Step-Level Checkpointing + Corrupt Checkpoint Quarantine**: Saves every `SAVE_STEPS` directly to Google Drive. If Colab disconnects *mid-save*, incomplete checkpoints are quarantined automatically and training resumes from the previous valid step.
- **Resumable Evaluation**: `scripts/run_eval.py` streams batch predictions to `eval_predictions_<split>.jsonl` with `os.fsync()`. If interrupted during evaluation, re-running **Cell 5** resumes from the exact remaining utterances.

In [ ]:
#@title 1. Mount Google Drive, Clone Repo & Install Dependencies { display-mode: "form" }
import os
import subprocess
import torch
from google.colab import drive

REPO_URL = "https://github.com/munzirds/nasr2.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
CLONE_DIR = "/content/nasr2" #@param {type:"string"}

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type and select A100 GPU or L4 GPU."
    )

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Clone or update the repository
if not os.path.exists(os.path.join(CLONE_DIR, "pyproject.toml")):
    print(f"--> Cloning {REPO_URL} into {CLONE_DIR}...")
    cmd = ["git", "clone", REPO_URL, CLONE_DIR]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{res.stderr}")
else:
    print(f"--> Pulling latest changes in {CLONE_DIR}...")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=False)

if REPO_BRANCH:
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", REPO_BRANCH], check=False)

%cd {CLONE_DIR}
!bash scripts/colab_setup.sh

In [ ]:
#@title 2. Configure Dataset Source (Hugging Face OR Google Drive Shards) & Hyperparameters { display-mode: "form" }
import os
import re
from getpass import getpass
from huggingface_hub import login
from kas_w2v_bert.data import discover_drive_shard_dirs, resolve_hf_token

# --- 1. Choose Dataset Source ---
DATASET_SOURCE = "huggingface" #@param ["huggingface", "gdrive", "auto"]

# --- Option A: Hugging Face Dataset Configuration ---
HF_DATASET_NAME = "munzirahangar/kas-news-dataset" #@param {type:"string"}
HF_DATASET_SUBSET = "" #@param {type:"string"}
HF_DATASET_SPLIT = "train" #@param {type:"string"}

# --- Option B: Google Drive Sharded Dataset Configuration ---
DRIVE_BASE_DIR = "/content/drive/MyDrive/nasr_kashmiri_asr" #@param {type:"string"}
DRIVE_DATASET_SUBFOLDER = "voicehub_dataset" #@param {type:"string"}
SHARD_GLOB = "batch_*" #@param {type:"string"}

# --- Column Names & Split Ratios (Auto-detects 'transcript' vs 'sentence') ---
AUDIO_COLUMN = "audio" #@param {type:"string"}
TEXT_COLUMN = "sentence" #@param {type:"string"}
VAL_RATIO = 0.05 #@param {type:"number"}
TEST_RATIO = 0.05 #@param {type:"number"}
REMOVE_DIACRITICS = False #@param {type:"boolean"}

# --- Model & Training Configuration ---
BASE_MODEL = "facebook/w2v-bert-2.0" #@param {type:"string"}
LEARNING_RATE = 5e-5 #@param {type:"number"}
NUM_TRAIN_EPOCHS = 10.0 #@param {type:"number"}
MAX_STEPS = -1 #@param {type:"integer"}
SAVE_STEPS = 200 #@param {type:"integer"}
EVAL_STEPS = 200 #@param {type:"integer"}
FREEZE_CONFORMER_LAYERS = 0 #@param {type:"integer"}

# --- Persistent Output Directory on Google Drive ---
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/nasr2_kashmiri_w2v_bert" #@param {type:"string"}
ISOLATE_BY_DATASET_NAME = True #@param {type:"boolean"}

# --- Optional Hugging Face Hub Backup ---
PUSH_TO_HUB = False #@param {type:"boolean"}
HUB_MODEL_ID = "" #@param {type:"string"}

# Resolve Google Drive dataset path if using gdrive
DRIVE_DS_DIR = (
    os.path.join(DRIVE_BASE_DIR, DRIVE_DATASET_SUBFOLDER)
    if DRIVE_DATASET_SUBFOLDER.strip()
    else DRIVE_BASE_DIR
)

# Authenticate with Hugging Face (needed for private HF datasets, base models, or Hub push)
hf_token = resolve_hf_token()
if not hf_token and (DATASET_SOURCE == "huggingface" or PUSH_TO_HUB):
    hf_token = getpass("Enter your Hugging Face token (or add HF_TOKEN in Colab Secrets): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print("[✓] Authenticated with Hugging Face.")

# Verify Google Drive shards immediately if DATASET_SOURCE == 'gdrive'
if DATASET_SOURCE == "gdrive":
    shard_dirs = discover_drive_shard_dirs(DRIVE_DS_DIR, shard_glob=SHARD_GLOB)
    print(f"[✓] Verified {len(shard_dirs)} valid dataset shard(s) in {DRIVE_DS_DIR}")
    active_ds_label = f"gdrive_{os.path.basename(DRIVE_DS_DIR.rstrip('/'))}"
else:
    active_ds_label = re.sub(r"[^a-zA-Z0-9_.-]+", "__", HF_DATASET_NAME.strip("/"))
    if HF_DATASET_SUBSET.strip():
        active_ds_label += f"__{HF_DATASET_SUBSET.strip()}"

mode_slug = "dg" if REMOVE_DIACRITICS else "da"
DRIVE_OUTPUT_DIR = (
    os.path.join(DRIVE_OUTPUT_ROOT, f"{active_ds_label}_{mode_slug}")
    if ISOLATE_BY_DATASET_NAME
    else DRIVE_OUTPUT_ROOT
)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print("=" * 72)
print("  CONFIGURED EXPERIMENT SUMMARY")
print("=" * 72)
print(f"  Dataset Source:       {DATASET_SOURCE}")
if DATASET_SOURCE == "gdrive":
    print(f"  Drive Dataset Path:   {DRIVE_DS_DIR} (shard_glob='{SHARD_GLOB}')")
else:
    print(f"  HF Dataset:           {HF_DATASET_NAME} (subset='{HF_DATASET_SUBSET or 'default'}', split='{HF_DATASET_SPLIT}')")
print(f"  Preferred Columns:    audio='{AUDIO_COLUMN}', text='{TEXT_COLUMN}' (auto-detects 'transcript'/'sentence')")
print(f"  Orthography Mode:     {'Diacritic-Agnostic (DG)' if REMOVE_DIACRITICS else 'Diacritic-Aware (DA)'}")
print(f"  Base Model:           {BASE_MODEL}")
print(f"  Persistent Drive Dir: {DRIVE_OUTPUT_DIR}")
print("=" * 72)

In [ ]:
#@title 3. Inspect Existing Checkpoints / Run Status on Google Drive { display-mode: "form" }
import json
import os
from kas_w2v_bert.checkpoint_manager import find_latest_valid_checkpoint, list_checkpoint_dirs_sorted

ckpts_dir = os.path.join(DRIVE_OUTPUT_DIR, "checkpoints")
status_file = os.path.join(DRIVE_OUTPUT_DIR, "run_status.json")

print("=" * 70)
print("  PERSISTENT CHECKPOINT & RUN STATUS DIAGNOSTICS")
print("=" * 70)
all_ckpts = list_checkpoint_dirs_sorted(ckpts_dir)
valid_ckpt = find_latest_valid_checkpoint(ckpts_dir, require_optimizer=True, quarantine_corrupted=False)
print(f"  Output Directory:           {DRIVE_OUTPUT_DIR}")
print(f"  Total Checkpoints on Drive: {len(all_ckpts)}")
print(f"  Latest Valid Checkpoint:    {valid_ckpt or 'None (Fresh Run)'}")

if os.path.isfile(status_file):
    with open(status_file, "r", encoding="utf-8") as f:
        st = json.load(f)
    print(f"  Last Recorded Status:       {st.get('status')}")
    print(f"  Global Step / Epoch:        step {st.get('global_step')} (epoch {st.get('epoch')})")
    print(f"  Best Validation DA-WER:     {st.get('best_da_wer')}%")
    print(f"  Last Updated (UTC):         {st.get('updated_at_utc')}")
print("=" * 70)

In [ ]:
#@title 4. Run or Resume Wav2Vec2-BERT 2.0 Fine-Tuning (`scripts/run_train.py`) { display-mode: "form" }
# Safe to re-run after any Colab disconnect/restart — automatically validates
# checkpoints in DRIVE_OUTPUT_DIR/checkpoints and resumes from the latest intact step.
import subprocess
import sys

train_cmd = [
    sys.executable,
    "scripts/run_train.py",
    "--config", "configs/w2v_bert_kas_news.yaml",
    "--model_name_or_path", BASE_MODEL,
    "--dataset_source", DATASET_SOURCE,
    "--dataset_name", HF_DATASET_NAME,
    "--dataset_split", HF_DATASET_SPLIT,
    "--shard_glob", SHARD_GLOB,
    "--audio_column", AUDIO_COLUMN,
    "--text_column", TEXT_COLUMN,
    "--val_ratio", str(VAL_RATIO),
    "--test_ratio", str(TEST_RATIO),
    "--remove_diacritics", str(REMOVE_DIACRITICS).lower(),
    "--output_dir", DRIVE_OUTPUT_DIR,
    "--learning_rate", str(LEARNING_RATE),
    "--num_train_epochs", str(NUM_TRAIN_EPOCHS),
    "--max_steps", str(MAX_STEPS),
    "--save_steps", str(SAVE_STEPS),
    "--eval_steps", str(EVAL_STEPS),
    "--freeze_conformer_layers", str(FREEZE_CONFORMER_LAYERS),
    "--push_to_hub", str(PUSH_TO_HUB).lower(),
]

if DATASET_SOURCE in ("gdrive", "auto") and DRIVE_DS_DIR:
    if DATASET_SOURCE == "gdrive":
        train_cmd.extend(["--drive_dataset_dir", DRIVE_DS_DIR])
if HF_DATASET_SUBSET.strip():
    train_cmd.extend(["--dataset_subset", HF_DATASET_SUBSET.strip()])
if PUSH_TO_HUB and HUB_MODEL_ID.strip():
    train_cmd.extend(["--hub_model_id", HUB_MODEL_ID.strip()])

print("Executing:", " ".join(train_cmd))
process = subprocess.Popen(train_cmd, stdout=sys.stdout, stderr=sys.stderr)
retcode = process.wait()
if retcode != 0:
    raise RuntimeError(f"Training exited with code {retcode}")

In [ ]:
#@title 5. Run Resumable Evaluation (`scripts/run_eval.py`) { display-mode: "form" }
# Streams batch predictions to DRIVE_OUTPUT_DIR/evaluation/eval_predictions_<EVAL_SPLIT>.jsonl.
# If interrupted mid-evaluation, re-running this cell resumes from the exact remaining utterances.
import subprocess
import sys

EVAL_SPLIT = "test" #@param ["test", "validation", "train"]
FORCE_RECOMPUTE = False #@param {type:"boolean"}

eval_cmd = [
    sys.executable,
    "scripts/run_eval.py",
    "--config", "configs/w2v_bert_kas_news.yaml",
    "--dataset_source", DATASET_SOURCE,
    "--dataset_name", HF_DATASET_NAME,
    "--dataset_split", HF_DATASET_SPLIT,
    "--shard_glob", SHARD_GLOB,
    "--audio_column", AUDIO_COLUMN,
    "--text_column", TEXT_COLUMN,
    "--val_ratio", str(VAL_RATIO),
    "--test_ratio", str(TEST_RATIO),
    "--remove_diacritics", str(REMOVE_DIACRITICS).lower(),
    "--output_dir", DRIVE_OUTPUT_DIR,
    "--eval_split", EVAL_SPLIT,
]

if DATASET_SOURCE == "gdrive" and DRIVE_DS_DIR:
    eval_cmd.extend(["--drive_dataset_dir", DRIVE_DS_DIR])
if HF_DATASET_SUBSET.strip():
    eval_cmd.extend(["--dataset_subset", HF_DATASET_SUBSET.strip()])
if FORCE_RECOMPUTE:
    eval_cmd.append("--force_recompute")

print("Executing:", " ".join(eval_cmd))
process = subprocess.Popen(eval_cmd, stdout=sys.stdout, stderr=sys.stderr)
retcode = process.wait()
if retcode != 0:
    raise RuntimeError(f"Evaluation exited with code {retcode}")

In [ ]:
#@title 6. Display Evaluation Report & Sample Transcriptions { display-mode: "form" }
import os
from IPython.display import Markdown, display

report_md_path = os.path.join(DRIVE_OUTPUT_DIR, "evaluation", f"eval_report_{EVAL_SPLIT}.md")
if os.path.isfile(report_md_path):
    with open(report_md_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print(f"Evaluation report not found at {report_md_path}. Run Cell 5 first.")